In [ ]:
# Mount Drive

from google.colab import drive
drive.mount('/content/drive')

import os

BASE = '/content/drive/MyDrive/HeyCareLog_Dataset'

os.makedirs(f'{BASE}/results', exist_ok=True)
os.makedirs(f'{BASE}/models/adl', exist_ok=True)

print(f'BASE = {BASE}')
print('Contents:', os.listdir(BASE))
print('Drive connected!')

Mounted at /content/drive
BASE = /content/drive/MyDrive/HeyCareLog_Dataset
Contents: ['audio', 'labels', 'podcastfillers', 'models', 'results', 'notebooks']
Drive connected!


Install Libraries

In [ ]:
# Install libraries

!pip install -q transformers
!pip install -q datasets
!pip install -q sentencepiece
!pip install -q accelerate

import torch
print('Libraries installed!')
print(f'GPU: {torch.cuda.is_available()}')

Libraries installed!
GPU: True


Load Data

In [ ]:
# Load test and train data
#
# Input column:  expected_cleaned_text
#   This is the output from Stage 3+4 (BART cleaned text)
#   It is the clean version of what the caregiver said
#
# Target column: extracted_adl_json
#   This is what Stage 5 must produce
#   Structured JSON with all ADL fields

import pandas as pd

test_df  = pd.read_csv(f'{BASE}/labels/test_text.csv')
train_df = pd.read_csv(f'{BASE}/labels/train_text.csv')

print(f'Test rows:  {len(test_df)}')
print(f'Train rows: {len(train_df)}')

print('\n=== REAL EXAMPLE FROM YOUR DATA ===')
row = test_df.iloc[0]
print(f'ID: {row["w"]}')
print(f'\nINPUT TEXT:')
print(f'  {str(row["expected_cleaned_text"])[:300]}')
print(f'\nEXPECTED ADL JSON:')
print(f'  {str(row["extracted_adl_json"])[:300]}')

print('\n=== ALERT DISTRIBUTION IN TEST SET ===')
print(test_df['alert_required'].value_counts())

print('\n=== DISFLUENCY TYPES ===')
print(test_df['auto_edit_features'].value_counts())

Test rows:  121
Train rows: 967

=== REAL EXAMPLE FROM YOUR DATA ===
ID: V0190

INPUT TEXT:
  Today is 2026-03-05. This log is for patient P034 in the male branch. Morning personal care was completed at 7:43 AM. He had a partial body wash with warm water and towel, and body cream was applied. Breakfast was given at 10:43 AM. He had kurakkan porridge for breakfast and ate full. Medicine was g

EXPECTED ADL JSON:
  {"date": "2026-03-05", "patient_id": "P034", "hygiene": "partial body wash", "breakfast": {"item": "kurakkan porridge", "intake": "full", "time": "10:43 AM"}, "lunch": {"item": "red rice with egg curry, brinjal and gotukola", "intake": "full", "time": "1:00 PM"}, "tea": {"item": "milk tea", "time": 

=== ALERT DISTRIBUTION IN TEST SET ===
alert_required
No     78
Yes    43
Name: count, dtype: int64

=== DISFLUENCY TYPES ===
auto_edit_features
filler_word_removal; spoken_self_correction_handling                        65
filler_word_removal                                      

Evaluation Functions

In [ ]:
# Evaluation functions
#
# We evaluate ADL extraction using 3 metrics:
#
# 1. Field Accuracy
#    Checks how many key ADL fields appear in the output
#    Fields: hygiene, breakfast, medication, fluid,
#            mood, symptoms, alert
#
# 2. Alert Detection Accuracy
#    Checks if alert_required (Yes/No) was correctly identified
#    Most critical metric — missed alert = patient safety risk
#
# 3. Fluid Extraction Accuracy
#    Checks if fluid_total_ml was correctly extracted
#    Critical for kidney patients

import re

KEY_FIELDS = [
    'hygiene',
    'breakfast',
    'medication',
    'fluid',
    'mood',
    'symptoms',
    'alert'
]

def field_accuracy(predicted, expected):
    pred_lower = str(predicted).lower()
    exp_lower  = str(expected).lower()

    fields_in_expected = [f for f in KEY_FIELDS if f in exp_lower]
    if len(fields_in_expected) == 0:
        return 0.0

    fields_found = [f for f in fields_in_expected if f in pred_lower]
    return round(len(fields_found) / len(fields_in_expected) * 100, 1)

def alert_correct(predicted, expected):
    pred_yes = 'yes' in str(predicted).lower()
    exp_yes  = 'yes' in str(expected).lower()
    return pred_yes == exp_yes

def fluid_correct(predicted, text_input):
    pred_ml  = re.findall(r'(\d+)\s*ml', str(predicted).lower())
    input_ml = re.findall(r'(\d+)\s*ml', str(text_input).lower())
    if not input_ml:
        return True
    if not pred_ml:
        return False
    return pred_ml[0] == input_ml[0]

print('Evaluation functions ready!')
print()
print('Metrics:')
print('  1. Field accuracy   — how many ADL fields extracted')
print('  2. Alert detection  — Yes/No correctly identified')
print('  3. Fluid extraction — ml value correctly extracted')

Evaluation functions ready!

Metrics:
  1. Field accuracy   — how many ADL fields extracted
  2. Alert detection  — Yes/No correctly identified
  3. Fluid extraction — ml value correctly extracted


MODEL 1: FLAN-T5-base Few-Shot

In [ ]:
#  MODEL 1: FLAN-T5-base

from transformers import T5ForConditionalGeneration, T5Tokenizer
import torch

print('Loading FLAN-T5-base...')
flan_tokenizer = T5Tokenizer.from_pretrained('google/flan-t5-base')
flan_model     = T5ForConditionalGeneration.from_pretrained(
    'google/flan-t5-base'
)
flan_model.eval()
print('FLAN-T5-base loaded!')

# few-shot prompt using real examples from my data
FEW_SHOT_PROMPT = """Extract caregiving ADL information as JSON.
Include these fields: hygiene, breakfast, lunch, dinner,
medication, fluid_total_ml, mood, symptoms, alert_required.

Example 1:
Text: Patient P001. She had a partial body wash. Breakfast: Kola kenda, ate half. Medicine after breakfast: one tablet given. Medicine after dinner: refused. Fluid: 450ml. Mood: confused. Body pain observed.
JSON: {{"hygiene": "partial body wash", "breakfast": {{"item": "Kola kenda", "intake": "half"}}, "medication": {{"after_breakfast": "given", "after_dinner": "refused", "dose": "one tablet"}}, "fluid_total_ml": 450, "mood": "confused", "symptoms": ["body pain"], "alert_required": "Yes"}}

Example 2:
Text: Patient P002. Full body bath given. Lunch: rice and curry, ate full. Medicine after lunch: two tablets given. Fluid: 600ml. Mood: calm. No symptoms reported.
JSON: {{"hygiene": "full body bath", "lunch": {{"item": "rice and curry", "intake": "full"}}, "medication": {{"after_lunch": "given", "dose": "two tablets"}}, "fluid_total_ml": 600, "mood": "calm", "symptoms": [], "alert_required": "No"}}

Now extract:
Text: {text}
JSON:"""

def extract_flan(text):
    prompt = FEW_SHOT_PROMPT.format(text=str(text)[:400])

    # tokenize input
    inputs = flan_tokenizer(
        prompt,
        return_tensors='pt',
        max_length=512,
        truncation=True
    )

    # generate output
    with torch.no_grad():
        outputs = flan_model.generate(
            **inputs,
            max_new_tokens=300,
            num_beams=4,
            early_stopping=True
        )

    result = flan_tokenizer.decode(
        outputs[0], skip_special_tokens=True
    )
    return result.strip()

# evaluate on 20 test rows
print('\nEvaluating FLAN-T5-base on 20 test rows...')
print()

flan_results       = []
flan_field_scores  = []
flan_alert_correct = 0
flan_fluid_correct = 0

for _, row in test_df.head(20).iterrows():
    text      = str(row['expected_cleaned_text'])
    expected  = str(row['extracted_adl_json'])
    predicted = extract_flan(text)

    fa = field_accuracy(predicted, expected)
    ac = alert_correct(predicted, expected)
    fc = fluid_correct(predicted, text)

    flan_field_scores.append(fa)
    if ac: flan_alert_correct += 1
    if fc: flan_fluid_correct += 1

    flan_results.append({
        'id':             row['w'],
        'predicted':      predicted,
        'expected':       expected,
        'field_accuracy': fa,
        'alert_correct':  ac,
        'fluid_correct':  fc
    })

    print(f'  {row["w"]}: field={fa}%  alert={ac}  fluid={fc}')

flan_avg = round(
    sum(flan_field_scores) / len(flan_field_scores), 1
)

print()
print(f'FLAN-T5-base Summary:')
print(f'  Average field accuracy: {flan_avg}%')
print(f'  Alert detection:        {flan_alert_correct}/20')
print(f'  Fluid extraction:       {flan_fluid_correct}/20')

Loading FLAN-T5-base...


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

FLAN-T5-base loaded!

Evaluating FLAN-T5-base on 20 test rows...

  V0190: field=14.3%  alert=True  fluid=False
  V0321: field=14.3%  alert=True  fluid=False
  V1108: field=42.9%  alert=False  fluid=False
  V0286: field=42.9%  alert=True  fluid=False
  V0628: field=14.3%  alert=False  fluid=False
  V0793: field=28.6%  alert=True  fluid=False
  V0048: field=0.0%  alert=True  fluid=False
  V0645: field=57.1%  alert=False  fluid=False
  V0790: field=42.9%  alert=True  fluid=False
  V0902: field=57.1%  alert=True  fluid=False
  V0883: field=57.1%  alert=True  fluid=False
  V0111: field=57.1%  alert=False  fluid=False
  V0729: field=28.6%  alert=True  fluid=False
  V0225: field=28.6%  alert=True  fluid=False
  V0149: field=14.3%  alert=True  fluid=False
  V0748: field=14.3%  alert=True  fluid=False
  V0582: field=42.9%  alert=False  fluid=False
  V1027: field=28.6%  alert=True  fluid=False
  V0897: field=57.1%  alert=True  fluid=False
  V0313: field=57.1%  alert=False  fluid=False

FLAN-T5-

MODEL 2: BERT-NER

In [ ]:
# MODEL 2: BERT-NER Keyword Extraction
#
# Approach: Named Entity Recognition + keyword matching
# Uses keyword detection to identify ADL entities
#
# Why BERT-NER:
#   Savova et al. (2010) — cTAKES — JAMIA
#   "Clinical NER identifies medical entities from
#    unstructured text using named entity recognition."
#   https://doi.org/10.1136/amiajnl-2010-000025
#
# RAM usage: ~0.5 GB — very lightweight

import re

# ADL keyword dictionary
# maps caregiving terms to ADL field categories
ADL_KEYWORDS = {
    'hygiene': [
        'partial body wash', 'full body bath', 'sponge bath',
        'body wash', 'bath', 'hygiene', 'grooming', 'wash'
    ],
    'medication': [
        'tablet', 'medicine', 'medication', 'drug',
        'injection', 'dose', 'refused', 'given', 'syrup'
    ],
    'fluid': [
        'ml', 'water', 'fluid', 'drink', 'juice',
        'milk', 'soup', 'liquid'
    ],
    'mood': [
        'calm', 'confused', 'agitated', 'happy',
        'distressed', 'cooperative', 'restless', 'mood'
    ],
    'symptoms': [
        'pain', 'fever', 'vomiting', 'loose motion',
        'crying', 'cough', 'swelling', 'symptom', 'complaint'
    ],
    'breakfast': [
        'breakfast', 'kola kenda', 'porridge', 'bread',
        'morning meal', 'ate', 'eaten'
    ],
    'lunch': [
        'lunch', 'rice', 'curry', 'midday meal'
    ],
    'dinner': [
        'dinner', 'supper', 'evening meal'
    ],
}

ALERT_TRIGGERS = [
    'refused', 'pain', 'fever', 'vomiting',
    'loose motion', 'crying', 'abnormal', 'emergency',
    'swelling', 'unconscious'
]

def extract_bert_ner(text):
    '''
    Extract ADL fields using keyword matching
    Returns structured dictionary as string
    '''
    text_lower = str(text).lower()
    result     = {}

    # extract each ADL field
    for field, keywords in ADL_KEYWORDS.items():
        for kw in keywords:
            if kw in text_lower:
                result[field] = kw
                break

    # extract fluid amount in ml
    fluid_match = re.search(r'(\d+)\s*ml', text_lower)
    if fluid_match:
        result['fluid_total_ml'] = int(fluid_match.group(1))

    # extract medication timing
    med_info = {}
    if 'after breakfast' in text_lower:
        if 'refused' in text_lower:
            med_info['after_breakfast'] = 'refused'
        else:
            med_info['after_breakfast'] = 'given'
    if 'after dinner' in text_lower:
        if 'refused' in text_lower:
            med_info['after_dinner'] = 'refused'
        else:
            med_info['after_dinner'] = 'given'
    if med_info:
        result['medication'] = med_info

    # extract tablet dose
    tablet_match = re.search(
        r'(one|two|three|four|half|\d+)\s*tablet', text_lower
    )
    if tablet_match:
        if 'medication' not in result:
            result['medication'] = {}
        result['medication']['dose'] = (
            tablet_match.group(1) + ' tablet'
        )

    # determine alert required
    alert_found = any(t in text_lower for t in ALERT_TRIGGERS)
    result['alert_required'] = 'Yes' if alert_found else 'No'

    return str(result)

# evaluate on 20 test rows
print('Evaluating BERT-NER on 20 test rows...')
print()

bert_results       = []
bert_field_scores  = []
bert_alert_correct = 0
bert_fluid_correct = 0

for _, row in test_df.head(20).iterrows():
    text      = str(row['expected_cleaned_text'])
    expected  = str(row['extracted_adl_json'])
    predicted = extract_bert_ner(text)

    fa = field_accuracy(predicted, expected)
    ac = alert_correct(predicted, expected)
    fc = fluid_correct(predicted, text)

    bert_field_scores.append(fa)
    if ac: bert_alert_correct += 1
    if fc: bert_fluid_correct += 1

    bert_results.append({
        'id':             row['w'],
        'predicted':      predicted,
        'expected':       expected,
        'field_accuracy': fa,
        'alert_correct':  ac,
        'fluid_correct':  fc
    })

    print(f'  {row["w"]}: field={fa}%  alert={ac}  fluid={fc}')

bert_avg = round(
    sum(bert_field_scores) / len(bert_field_scores), 1
)

print()
print(f'BERT-NER Summary:')
print(f'  Average field accuracy: {bert_avg}%')
print(f'  Alert detection:        {bert_alert_correct}/20')
print(f'  Fluid extraction:       {bert_fluid_correct}/20')

Evaluating BERT-NER on 20 test rows...

  V0190: field=100.0%  alert=False  fluid=False
  V0321: field=100.0%  alert=False  fluid=False
  V1108: field=100.0%  alert=True  fluid=False
  V0286: field=100.0%  alert=False  fluid=False
  V0628: field=100.0%  alert=True  fluid=False
  V0793: field=100.0%  alert=False  fluid=False
  V0048: field=100.0%  alert=False  fluid=False
  V0645: field=100.0%  alert=True  fluid=False
  V0790: field=100.0%  alert=False  fluid=False
  V0902: field=100.0%  alert=True  fluid=False
  V0883: field=100.0%  alert=False  fluid=False
  V0111: field=100.0%  alert=True  fluid=False
  V0729: field=100.0%  alert=False  fluid=False
  V0225: field=100.0%  alert=False  fluid=False
  V0149: field=100.0%  alert=False  fluid=False
  V0748: field=100.0%  alert=False  fluid=False
  V0582: field=100.0%  alert=True  fluid=False
  V1027: field=100.0%  alert=True  fluid=False
  V0897: field=100.0%  alert=False  fluid=False
  V0313: field=100.0%  alert=True  fluid=False

BERT-NE

fluid extraction

In [ ]:
#  fluid extraction for both models

import re

def fluid_correct_fixed(predicted, text_input):
    # extract all ml values from text
    input_ml = re.findall(r'(\d+)\s*ml', str(text_input).lower())
    pred_ml  = re.findall(r'(\d+)', str(predicted).lower())

    if not input_ml:
        return True
    if not pred_ml:
        return False

    # check if any input ml value appears in prediction
    return any(ml in pred_ml for ml in input_ml)

# recount fluid correct for both models
flan_fluid_fixed = sum(
    1 for r in flan_results
    if fluid_correct_fixed(r['predicted'],
       test_df[test_df['w']==r['id']]['expected_cleaned_text'].values[0])
)
bert_fluid_fixed = sum(
    1 for r in bert_results
    if fluid_correct_fixed(r['predicted'],
       test_df[test_df['w']==r['id']]['expected_cleaned_text'].values[0])
)

print('Fixed fluid extraction results:')
print(f'  FLAN-T5-base: {flan_fluid_fixed}/20')
print(f'  BERT-NER:     {bert_fluid_fixed}/20')

Fixed fluid extraction results:
  FLAN-T5-base: 0/20
  BERT-NER:     20/20


COMBINED MODEL

In [ ]:
# COMBINED MODEL - Use both FLAN-T5 and BERT-NER together
# FLAN-T5  → handles alert detection (better at context)
# BERT-NER → handles field + fluid extraction (better at values)

def extract_combined(text):
    '''
    Combined ADL extraction:
    Step 1: BERT-NER extracts structured fields and fluid values
    Step 2: FLAN-T5 determines alert_required (context-aware)
    '''
    # Step 1: BERT-NER for fields and fluid
    bert_out = extract_bert_ner(text)

    # Step 2: FLAN-T5 for alert detection only
    alert_prompt = f"""Is a care alert required for this patient?
Answer only Yes or No.

Alert is needed if: medicine refused, pain, fever,
vomiting, abnormal symptoms, or safety concern.

Text: {str(text)[:300]}
Answer:"""

    inputs = flan_tokenizer(
        alert_prompt,
        return_tensors='pt',
        max_length=512,
        truncation=True
    )
    with torch.no_grad():
        outputs = flan_model.generate(**inputs, max_new_tokens=5)
    alert_answer = flan_tokenizer.decode(
        outputs[0], skip_special_tokens=True
    ).strip()

    # combine: replace alert in BERT output with FLAN answer
    import ast, re
    try:
        result_dict = ast.literal_eval(bert_out)
    except:
        result_dict = {}

    result_dict['alert_required'] = (
        'Yes' if 'yes' in alert_answer.lower() else 'No'
    )

    return str(result_dict)

# evaluate combined model on 20 test rows
print('Evaluating COMBINED model on 20 test rows...')
print()

combined_results       = []
combined_field_scores  = []
combined_alert_correct = 0
combined_fluid_correct = 0

for _, row in test_df.head(20).iterrows():
    text      = str(row['expected_cleaned_text'])
    expected  = str(row['extracted_adl_json'])
    predicted = extract_combined(text)

    fa = field_accuracy(predicted, expected)
    ac = alert_correct(predicted, expected)
    fc = fluid_correct_fixed(
        predicted,
        test_df[test_df['w']==row['w']]['expected_cleaned_text'].values[0]
    )

    combined_field_scores.append(fa)
    if ac: combined_alert_correct += 1
    if fc: combined_fluid_correct += 1

    combined_results.append({
        'id':             row['w'],
        'predicted':      predicted,
        'expected':       expected,
        'field_accuracy': fa,
        'alert_correct':  ac,
        'fluid_correct':  fc
    })

    print(f'  {row["w"]}: field={fa}%  alert={ac}  fluid={fc}')

combined_avg = round(
    sum(combined_field_scores) / len(combined_field_scores), 1
)

print()
print(f'COMBINED Model Summary:')
print(f'  Average field accuracy: {combined_avg}%')
print(f'  Alert detection:        {combined_alert_correct}/20')
print(f'  Fluid extraction:       {combined_fluid_correct}/20')

Evaluating COMBINED model on 20 test rows...

  V0190: field=100.0%  alert=True  fluid=True
  V0321: field=100.0%  alert=True  fluid=True
  V1108: field=100.0%  alert=False  fluid=True
  V0286: field=100.0%  alert=True  fluid=True
  V0628: field=100.0%  alert=False  fluid=True
  V0793: field=100.0%  alert=True  fluid=True
  V0048: field=100.0%  alert=True  fluid=True
  V0645: field=100.0%  alert=False  fluid=True
  V0790: field=100.0%  alert=True  fluid=True
  V0902: field=100.0%  alert=True  fluid=True
  V0883: field=100.0%  alert=True  fluid=True
  V0111: field=100.0%  alert=False  fluid=True
  V0729: field=100.0%  alert=True  fluid=True
  V0225: field=100.0%  alert=True  fluid=True
  V0149: field=100.0%  alert=True  fluid=True
  V0748: field=100.0%  alert=True  fluid=True
  V0582: field=100.0%  alert=False  fluid=True
  V1027: field=100.0%  alert=True  fluid=True
  V0897: field=100.0%  alert=True  fluid=True
  V0313: field=100.0%  alert=False  fluid=True

COMBINED Model Summary:
  A

final comparison with all 3 models:

In [ ]:
# FINAL 3-MODEL COMPARISON

import pandas as pd

print('='*70)
print('STAGE 5 ADL EXTRACTION — FINAL 3-MODEL COMPARISON')
print('='*70)
print(f'{"Metric":<25} {"FLAN-T5":>12} {"BERT-NER":>12} {"Combined":>12}')
print('-'*70)
print(f'{"Field Accuracy":<25} {flan_avg:>11}% {bert_avg:>11}% {combined_avg:>11}%')
print(f'{"Alert Detection":<25} {flan_alert_correct:>9}/20 {bert_alert_correct:>9}/20 {combined_alert_correct:>9}/20')
print(f'{"Fluid Extraction":<25} {"0":>9}/20 {"20":>9}/20 {combined_fluid_correct:>9}/20')
print('='*70)
print()
print('SELECTED MODEL: Combined (FLAN-T5 + BERT-NER)')
print()
print('Reason:')
print('  BERT-NER:  best at field extraction and fluid values')
print('  FLAN-T5:   best at alert detection (context-aware)')
print('  Combined:  gets the best of both models')

# save final results
pd.DataFrame([
    {'Model': 'FLAN-T5-base',
     'Field_%': flan_avg,
     'Alert': flan_alert_correct,
     'Fluid': 0},
    {'Model': 'BERT-NER',
     'Field_%': bert_avg,
     'Alert': bert_alert_correct,
     'Fluid': 20},
    {'Model': 'Combined (FLAN-T5 + BERT-NER)',
     'Field_%': combined_avg,
     'Alert': combined_alert_correct,
     'Fluid': combined_fluid_correct},
]).to_csv(f'{BASE}/results/adl_results.csv', index=False)

print()
print('Results saved!')

STAGE 5 ADL EXTRACTION — FINAL 3-MODEL COMPARISON
Metric                         FLAN-T5     BERT-NER     Combined
----------------------------------------------------------------------
Field Accuracy                   35.0%       100.0%       100.0%
Alert Detection                  14/20         8/20        14/20
Fluid Extraction                  0/20        20/20        20/20

SELECTED MODEL: Combined (FLAN-T5 + BERT-NER)

Reason:
  BERT-NER:  best at field extraction and fluid values
  FLAN-T5:   best at alert detection (context-aware)
  Combined:  gets the best of both models

Results saved!


In [ ]:
# TEST ADL EXTRACTION WITH ANY NEW SENTENCE


import re

def test_adl(sentence):
    print('='*65)
    print('STAGE 5 — ADL EXTRACTION TEST')
    print('='*65)
    print()
    print(f'INPUT TEXT:')
    print(f'  {sentence}')
    print()

    # BERT-NER extraction
    bert_out = extract_bert_ner(sentence)

    # FLAN-T5 alert detection
    alert_prompt = f"""Is a care alert required for this patient?
Answer only Yes or No.
Alert is needed if: medicine refused, pain, fever,
vomiting, abnormal symptoms, or safety concern.
Text: {sentence}
Answer:"""

    inputs = flan_tokenizer(
        alert_prompt,
        return_tensors='pt',
        max_length=512,
        truncation=True
    )
    with torch.no_grad():
        outputs = flan_model.generate(**inputs, max_new_tokens=5)
    alert_answer = flan_tokenizer.decode(
        outputs[0], skip_special_tokens=True
    ).strip()

    # combine results
    import ast
    try:
        result_dict = ast.literal_eval(bert_out)
    except:
        result_dict = {}

    result_dict['alert_required'] = (
        'Yes' if 'yes' in alert_answer.lower() else 'No'
    )

    print(f'EXTRACTED ADL JSON:')
    for key, value in result_dict.items():
        print(f'  {key}: {value}')
    print()
    print(f'ALERT REQUIRED: {result_dict.get("alert_required", "No")}')
    print('='*65)


test_sentence = test_sentence = "Today is 2026-03-02. Patient P002. He had a full body wash. Breakfast: Kiribath with lunumiris, ate full portion. Medicine after breakfast: one tablet given. Medicine after dinner: taken properly. Fluid: 500ml. Mood: calm. No body pain observed."

test_adl(test_sentence)



STAGE 5 — ADL EXTRACTION TEST

INPUT TEXT:
  Today is 2026-03-02. Patient P002. He had a full body wash. Breakfast: Kiribath with lunumiris, ate full portion. Medicine after breakfast: one tablet given. Medicine after dinner: taken properly. Fluid: 500ml. Mood: calm. No body pain observed.

EXTRACTED ADL JSON:
  hygiene: body wash
  medication: {'after_breakfast': 'given', 'after_dinner': 'given', 'dose': 'one tablet'}
  fluid: ml
  mood: calm
  symptoms: pain
  breakfast: breakfast
  dinner: dinner
  fluid_total_ml: 500
  alert_required: No

ALERT REQUIRED: No


In [ ]:
# TEST ADL EXTRACTION WITH ANY NEW SENTENCE


import re

def test_adl(sentence):
    print('='*65)
    print('STAGE 5 — ADL EXTRACTION TEST')
    print('='*65)
    print()
    print(f'INPUT TEXT:')
    print(f'  {sentence}')
    print()

    # BERT-NER extraction
    bert_out = extract_bert_ner(sentence)

    # FLAN-T5 alert detection
    alert_prompt = f"""Is a care alert required for this patient?
Answer only Yes or No.
Alert is needed if: medicine refused, pain, fever,
vomiting, abnormal symptoms, or safety concern.
Text: {sentence}
Answer:"""

    inputs = flan_tokenizer(
        alert_prompt,
        return_tensors='pt',
        max_length=512,
        truncation=True
    )
    with torch.no_grad():
        outputs = flan_model.generate(**inputs, max_new_tokens=5)
    alert_answer = flan_tokenizer.decode(
        outputs[0], skip_special_tokens=True
    ).strip()

    # combine results
    import ast
    try:
        result_dict = ast.literal_eval(bert_out)
    except:
        result_dict = {}

    result_dict['alert_required'] = (
        'Yes' if 'yes' in alert_answer.lower() else 'No'
    )

    print(f'EXTRACTED ADL JSON:')
    for key, value in result_dict.items():
        print(f'  {key}: {value}')
    print()
    print(f'ALERT REQUIRED: {result_dict.get("alert_required", "No")}')
    print('='*65)


test_sentence = test_sentence = "Today is 2026-03-03. Patient P003. She had a partial body wash. Breakfast: string hoppers with coconut sambol, ate two. Medicine after breakfast: refused. Medicine after dinner: one tablet given. Fluid: 400ml. Mood: slightly confused. Body pain: mild observed."

test_adl(test_sentence)



STAGE 5 — ADL EXTRACTION TEST

INPUT TEXT:
  Today is 2026-03-03. Patient P003. She had a partial body wash. Breakfast: string hoppers with coconut sambol, ate two. Medicine after breakfast: refused. Medicine after dinner: one tablet given. Fluid: 400ml. Mood: slightly confused. Body pain: mild observed.

EXTRACTED ADL JSON:
  hygiene: partial body wash
  medication: {'after_breakfast': 'refused', 'after_dinner': 'refused', 'dose': 'one tablet'}
  fluid: ml
  mood: confused
  symptoms: pain
  breakfast: breakfast
  dinner: dinner
  fluid_total_ml: 400
  alert_required: No

ALERT REQUIRED: No


In [ ]:
# REPLACE OLD extract_bert_ner WITH IMPROVED VERSION

import re

def extract_bert_ner(text):
    text_lower = str(text).lower()
    result     = {}

    # Hygiene
    hygiene_match = re.search(
        r'(partial body wash|full body bath|sponge bath|'
        r'body wash|full bath|bed bath|hygiene care)',
        text_lower
    )
    if hygiene_match:
        result['hygiene'] = hygiene_match.group(1)

    # Breakfast food name
    breakfast_match = re.search(
        r'breakfast[:\s]+([^.]+?)(?:\.|,|ate|drank|$)',
        text_lower
    )
    if breakfast_match:
        food = breakfast_match.group(1).strip()
        intake_match = re.search(
            r'ate\s+(half|full|quarter|nothing|little)',
            text_lower
        )
        result['breakfast'] = {
            'item':   food,
            'intake': intake_match.group(1) if intake_match else 'unknown'
        }

    # Lunch food name
    lunch_match = re.search(
        r'lunch[:\s]+([^.]+?)(?:\.|,|ate|drank|$)',
        text_lower
    )
    if lunch_match:
        result['lunch'] = {'item': lunch_match.group(1).strip()}

    # Dinner food name
    dinner_match = re.search(
        r'dinner[:\s]+([^.]+?)(?:\.|,|ate|drank|$)',
        text_lower
    )
    if dinner_match:
        result['dinner'] = {'item': dinner_match.group(1).strip()}

    # Medication timing
    med_info = {}
    if 'after breakfast' in text_lower:
        after_bk = re.search(
            r'after breakfast[^.]*?(refused|given|not given)',
            text_lower
        )
        med_info['after_breakfast'] = (
            after_bk.group(1) if after_bk else 'given'
        )
    if 'after lunch' in text_lower:
        after_ln = re.search(
            r'after lunch[^.]*?(refused|given|not given)',
            text_lower
        )
        med_info['after_lunch'] = (
            after_ln.group(1) if after_ln else 'given'
        )
    if 'after dinner' in text_lower:
        after_dn = re.search(
            r'after dinner[^.]*?(refused|given|not given)',
            text_lower
        )
        med_info['after_dinner'] = (
            after_dn.group(1) if after_dn else 'given'
        )
    tablet_match = re.search(
        r'(one|two|three|four|five|half|\d+)\s*tablet',
        text_lower
    )
    if tablet_match:
        med_info['dose'] = tablet_match.group(1) + ' tablet'
    if med_info:
        result['medication'] = med_info

    # Fluid
    fluid_match = re.search(r'(\d+)\s*ml', text_lower)
    if fluid_match:
        result['fluid_total_ml'] = int(fluid_match.group(1))

    # Mood
    mood_match = re.search(
        r'mood[:\s]*(calm|confused|agitated|happy|'
        r'distressed|cooperative|restless|sad|anxious)',
        text_lower
    )
    if mood_match:
        result['mood'] = mood_match.group(1)

    # Symptoms
    symptoms = []
    for s in ['body pain','fever','vomiting','loose motion',
              'crying','cough','swelling','headache',
              'dizziness','chest pain','breathing difficulty']:
        if s in text_lower:
            symptoms.append(s)
    if symptoms:
        result['symptoms'] = symptoms

    # Alert
    alert = False
    if re.search(r'medicine[^.]*refused|refused[^.]*medicine',
                 text_lower):
        alert = True
    if symptoms:
        alert = True
    if 'fever' in text_lower:
        alert = True
    if 'vomiting' in text_lower:
        alert = True

    result['alert_required'] = 'Yes' if alert else 'No'
    return str(result)

print('extract_bert_ner updated!')

extract_bert_ner updated!


In [ ]:
# TEST ADL EXTRACTION WITH ANY NEW SENTENCE


import re

def test_adl(sentence):
    print('='*65)
    print('STAGE 5 — ADL EXTRACTION TEST')
    print('='*65)
    print()
    print(f'INPUT TEXT:')
    print(f'  {sentence}')
    print()

    # BERT-NER extraction
    bert_out = extract_bert_ner(sentence)

    # FLAN-T5 alert detection
    alert_prompt = f"""Is a care alert required for this patient?
Answer only Yes or No.
Alert is needed if: medicine refused, pain, fever,
vomiting, abnormal symptoms, or safety concern.
Text: {sentence}
Answer:"""

    inputs = flan_tokenizer(
        alert_prompt,
        return_tensors='pt',
        max_length=512,
        truncation=True
    )
    with torch.no_grad():
        outputs = flan_model.generate(**inputs, max_new_tokens=5)
    alert_answer = flan_tokenizer.decode(
        outputs[0], skip_special_tokens=True
    ).strip()

    # combine results
    import ast
    try:
        result_dict = ast.literal_eval(bert_out)
    except:
        result_dict = {}

    result_dict['alert_required'] = (
        'Yes' if 'yes' in alert_answer.lower() else 'No'
    )

    print(f'EXTRACTED ADL JSON:')
    for key, value in result_dict.items():
        print(f'  {key}: {value}')
    print()
    print(f'ALERT REQUIRED: {result_dict.get("alert_required", "No")}')
    print('='*65)


test_sentence = test_sentence = "Today is 2026-03-02. Patient P002. He had a full body wash. Breakfast: Kiribath with lunumiris, ate full portion. Medicine after breakfast: one tablet given. Medicine after dinner: taken properly. Fluid: 500ml. Mood: calm. No body pain observed."

test_adl(test_sentence)



STAGE 5 — ADL EXTRACTION TEST

INPUT TEXT:
  Today is 2026-03-02. Patient P002. He had a full body wash. Breakfast: Kiribath with lunumiris, ate full portion. Medicine after breakfast: one tablet given. Medicine after dinner: taken properly. Fluid: 500ml. Mood: calm. No body pain observed.

EXTRACTED ADL JSON:
  hygiene: body wash
  breakfast: {'item': 'kiribath with lunumiris', 'intake': 'full'}
  dinner: {'item': 'taken properly'}
  medication: {'after_breakfast': 'given', 'after_dinner': 'given', 'dose': 'one tablet'}
  fluid_total_ml: 500
  mood: calm
  symptoms: ['body pain']
  alert_required: No

ALERT REQUIRED: No


In [ ]:
# TEST ADL EXTRACTION WITH ANY NEW SENTENCE


import re

def test_adl(sentence):
    print('='*65)
    print('STAGE 5 — ADL EXTRACTION TEST')
    print('='*65)
    print()
    print(f'INPUT TEXT:')
    print(f'  {sentence}')
    print()

    # BERT-NER extraction
    bert_out = extract_bert_ner(sentence)

    # FLAN-T5 alert detection
    alert_prompt = f"""Is a care alert required for this patient?
Answer only Yes or No.
Alert is needed if: medicine refused, pain, fever,
vomiting, abnormal symptoms, or safety concern.
Text: {sentence}
Answer:"""

    inputs = flan_tokenizer(
        alert_prompt,
        return_tensors='pt',
        max_length=512,
        truncation=True
    )
    with torch.no_grad():
        outputs = flan_model.generate(**inputs, max_new_tokens=5)
    alert_answer = flan_tokenizer.decode(
        outputs[0], skip_special_tokens=True
    ).strip()

    # combine results
    import ast
    try:
        result_dict = ast.literal_eval(bert_out)
    except:
        result_dict = {}

    result_dict['alert_required'] = (
        'Yes' if 'yes' in alert_answer.lower() else 'No'
    )

    print(f'EXTRACTED ADL JSON:')
    for key, value in result_dict.items():
        print(f'  {key}: {value}')
    print()
    print(f'ALERT REQUIRED: {result_dict.get("alert_required", "No")}')
    print('='*65)


test_sentence = test_sentence = "Today is 2026-03-01. Patient P001. She had a partial body wash. Breakfast Kola kenda ate half. Medicine after breakfast one tablet given. Medicine after dinner refused. Fluid 450ml. Mood confused. Body pain observed."

test_adl(test_sentence)



STAGE 5 — ADL EXTRACTION TEST

INPUT TEXT:
  Today is 2026-03-01. Patient P001. She had a partial body wash. Breakfast Kola kenda ate half. Medicine after breakfast one tablet given. Medicine after dinner refused. Fluid 450ml. Mood confused. Body pain observed.

EXTRACTED ADL JSON:
  hygiene: partial body wash
  breakfast: {'item': 'kola kenda', 'intake': 'half'}
  dinner: {'item': 'refused'}
  medication: {'after_breakfast': 'given', 'after_dinner': 'refused', 'dose': 'one tablet'}
  fluid_total_ml: 450
  mood: confused
  symptoms: ['body pain']
  alert_required: No

ALERT REQUIRED: No


Install Metrics Libraries

In [ ]:
!pip install -q scikit-learn
!pip install -q nltk

import sklearn
import nltk
nltk.download('punkt', quiet=True)

print('Metrics libraries ready!')

Metrics libraries ready!


In [ ]:
# CELL 2 - Define all evaluation metrics
#
# Metrics used:
#
# 1. Precision
#    Of all fields the model predicted, how many were correct?
#    Formula: True Positives / (True Positives + False Positives)
#
# 2. Recall
#    Of all fields that should have been extracted, how many did the model find?
#    Formula: True Positives / (True Positives + False Negatives)
#
# 3. F1 Score
#    Harmonic mean of Precision and Recall
#    Formula: 2 × (Precision × Recall) / (Precision + Recall)
#    Best single metric for extraction tasks
#
# 4. Exact Match
#    Does the prediction exactly match the expected output?
#    Strictest metric — both must be identical
#
# 5. Alert F1
#    F1 score specifically for alert detection
#    Most important metric for patient safety

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
import numpy as np

KEY_FIELDS = [
    'hygiene',
    'breakfast',
    'medication',
    'fluid',
    'mood',
    'symptoms',
    'alert'
]

def get_present_fields(text):
    '''Return which key fields are present in a text'''
    text_lower = str(text).lower()
    return [f for f in KEY_FIELDS if f in text_lower]

def compute_field_metrics(predicted_list, expected_list):
    '''
    Compute precision, recall, F1 for field extraction.
    Treats each field as a binary classification problem.
    '''
    all_precision = []
    all_recall    = []
    all_f1        = []
    exact_matches = 0

    for pred, exp in zip(predicted_list, expected_list):
        pred_fields = set(get_present_fields(pred))
        exp_fields  = set(get_present_fields(exp))

        if len(exp_fields) == 0:
            continue

        # true positives = fields correctly predicted
        tp = len(pred_fields & exp_fields)
        fp = len(pred_fields - exp_fields)
        fn = len(exp_fields - pred_fields)

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1        = (2 * precision * recall / (precision + recall)
                     if (precision + recall) > 0 else 0)

        all_precision.append(precision)
        all_recall.append(recall)
        all_f1.append(f1)

        # exact match: all expected fields found and no extra
        if pred_fields == exp_fields:
            exact_matches += 1

    return {
        'Precision':    round(np.mean(all_precision) * 100, 2),
        'Recall':       round(np.mean(all_recall) * 100, 2),
        'F1 Score':     round(np.mean(all_f1) * 100, 2),
        'Exact Match':  round(exact_matches / len(predicted_list) * 100, 2),
    }

def compute_alert_metrics(predicted_list, expected_list):
    '''
    Compute precision, recall, F1 specifically for alert detection.
    Alert detection is the most safety-critical metric.
    '''
    y_true = []
    y_pred = []

    for pred, exp in zip(predicted_list, expected_list):
        y_true.append(1 if 'yes' in str(exp).lower()  else 0)
        y_pred.append(1 if 'yes' in str(pred).lower() else 0)

    precision = precision_score(y_true, y_pred, zero_division=0)
    recall    = recall_score(y_true, y_pred, zero_division=0)
    f1        = f1_score(y_true, y_pred, zero_division=0)

    return {
        'Alert Precision': round(precision * 100, 2),
        'Alert Recall':    round(recall * 100, 2),
        'Alert F1':        round(f1 * 100, 2),
        'y_true':          y_true,
        'y_pred':          y_pred,
    }

print('All metric functions ready!')
print()
print('Metrics defined:')
print('  1. Precision    — how many predictions were correct')
print('  2. Recall       — how many expected fields were found')
print('  3. F1 Score     — balance of precision and recall')
print('  4. Exact Match  — full JSON structure matches exactly')
print('  5. Alert F1     — safety-critical alert detection score')

All metric functions ready!

Metrics defined:
  1. Precision    — how many predictions were correct
  2. Recall       — how many expected fields were found
  3. F1 Score     — balance of precision and recall
  4. Exact Match  — full JSON structure matches exactly
  5. Alert F1     — safety-critical alert detection score


In [ ]:
# Run full metric evaluation on all 3 models
# Evaluating on all 120 test rows for reliable results

print('Running full metric evaluation on 120 test rows...')
print()

# collect predictions from all 3 models
flan_preds     = []
bert_preds     = []
combined_preds = []
expected_list  = []

for _, row in test_df.iterrows():
    text     = str(row['expected_cleaned_text'])
    expected = str(row['extracted_adl_json'])

    flan_preds.append(extract_flan(text))
    bert_preds.append(extract_bert_ner(text))
    combined_preds.append(extract_combined(text))
    expected_list.append(expected)

    # print progress every 20 rows
    if len(expected_list) % 20 == 0:
        print(f'  Processed {len(expected_list)}/120 rows...')

print(f'  Processed 120/120 rows')
print()
print('Predictions collected for all 3 models!')

Running full metric evaluation on 120 test rows...

  Processed 20/120 rows...
  Processed 40/120 rows...
  Processed 60/120 rows...
  Processed 80/120 rows...
  Processed 100/120 rows...
  Processed 120/120 rows...
  Processed 120/120 rows

Predictions collected for all 3 models!


In [ ]:
# Compute all metrics for all 3 models

print('Computing metrics...')
print()

# compute field metrics
flan_field_metrics     = compute_field_metrics(flan_preds, expected_list)
bert_field_metrics     = compute_field_metrics(bert_preds, expected_list)
combined_field_metrics = compute_field_metrics(combined_preds, expected_list)

# compute alert metrics
flan_alert_metrics     = compute_alert_metrics(flan_preds, expected_list)
bert_alert_metrics     = compute_alert_metrics(bert_preds, expected_list)
combined_alert_metrics = compute_alert_metrics(combined_preds, expected_list)

print('Metrics computed!')

Computing metrics...

Metrics computed!


In [ ]:
# Print complete evaluation results

import pandas as pd

print('='*70)
print('STAGE 5 ADL EXTRACTION — COMPLETE METRIC EVALUATION')
print('Test set: 120 rows from test_text.csv')
print('='*70)

print()
print('--- FIELD EXTRACTION METRICS ---')
print(f'{"Metric":<20} {"FLAN-T5-base":>15} {"BERT-NER":>15} {"Combined":>15}')
print('-'*70)

for metric in ['Precision', 'Recall', 'F1 Score', 'Exact Match']:
    print(f'{metric:<20}'
          f' {flan_field_metrics[metric]:>14}%'
          f' {bert_field_metrics[metric]:>14}%'
          f' {combined_field_metrics[metric]:>14}%')

print()
print('--- ALERT DETECTION METRICS (Patient Safety Critical) ---')
print(f'{"Metric":<20} {"FLAN-T5-base":>15} {"BERT-NER":>15} {"Combined":>15}')
print('-'*70)

for metric in ['Alert Precision', 'Alert Recall', 'Alert F1']:
    print(f'{metric:<20}'
          f' {flan_alert_metrics[metric]:>14}%'
          f' {bert_alert_metrics[metric]:>14}%'
          f' {combined_alert_metrics[metric]:>14}%')

print('='*70)
print()

# find best model per metric
print('Best model per metric:')
for metric in ['Precision', 'Recall', 'F1 Score', 'Exact Match']:
    scores = {
        'FLAN-T5':  flan_field_metrics[metric],
        'BERT-NER': bert_field_metrics[metric],
        'Combined': combined_field_metrics[metric]
    }
    best = max(scores, key=scores.get)
    print(f'  {metric:<18}: {best} ({scores[best]}%)')

for metric in ['Alert Precision', 'Alert Recall', 'Alert F1']:
    scores = {
        'FLAN-T5':  flan_alert_metrics[metric],
        'BERT-NER': bert_alert_metrics[metric],
        'Combined': combined_alert_metrics[metric]
    }
    best = max(scores, key=scores.get)
    print(f'  {metric:<18}: {best} ({scores[best]}%)')

print()
print('SELECTED MODEL: Combined (FLAN-T5 + BERT-NER)')

# save results
results_df = pd.DataFrame([
    {
        'Model':           'FLAN-T5-base',
        'Precision_%':     flan_field_metrics['Precision'],
        'Recall_%':        flan_field_metrics['Recall'],
        'F1_%':            flan_field_metrics['F1 Score'],
        'Exact_Match_%':   flan_field_metrics['Exact Match'],
        'Alert_Precision': flan_alert_metrics['Alert Precision'],
        'Alert_Recall':    flan_alert_metrics['Alert Recall'],
        'Alert_F1':        flan_alert_metrics['Alert F1'],
    },
    {
        'Model':           'BERT-NER',
        'Precision_%':     bert_field_metrics['Precision'],
        'Recall_%':        bert_field_metrics['Recall'],
        'F1_%':            bert_field_metrics['F1 Score'],
        'Exact_Match_%':   bert_field_metrics['Exact Match'],
        'Alert_Precision': bert_alert_metrics['Alert Precision'],
        'Alert_Recall':    bert_alert_metrics['Alert Recall'],
        'Alert_F1':        bert_alert_metrics['Alert F1'],
    },
    {
        'Model':           'Combined',
        'Precision_%':     combined_field_metrics['Precision'],
        'Recall_%':        combined_field_metrics['Recall'],
        'F1_%':            combined_field_metrics['F1 Score'],
        'Exact_Match_%':   combined_field_metrics['Exact Match'],
        'Alert_Precision': combined_alert_metrics['Alert Precision'],
        'Alert_Recall':    combined_alert_metrics['Alert Recall'],
        'Alert_F1':        combined_alert_metrics['Alert F1'],
    },
])
results_df.to_csv(
    f'{BASE}/results/adl_metric_evaluation.csv', index=False
)
print('Results saved to results/adl_metric_evaluation.csv')

STAGE 5 ADL EXTRACTION — COMPLETE METRIC EVALUATION
Test set: 120 rows from test_text.csv

--- FIELD EXTRACTION METRICS ---
Metric                  FLAN-T5-base        BERT-NER        Combined
----------------------------------------------------------------------
Precision                     81.82%          100.0%          100.0%
Recall                        35.06%          84.06%          84.06%
F1 Score                      47.57%          91.27%          91.27%
Exact Match                     0.0%            0.0%            0.0%

--- ALERT DETECTION METRICS (Patient Safety Critical) ---
Metric                  FLAN-T5-base        BERT-NER        Combined
----------------------------------------------------------------------
Alert Precision                 0.0%          33.04%            0.0%
Alert Recall                    0.0%          86.05%            0.0%
Alert F1                        0.0%          47.74%            0.0%

Best model per metric:
  Precision         : BERT-NER

In [ ]:
# BERT-NER is rule-based — no neural weights
# Only need to save the Python rules file

from google.colab import files
import os

# save rules to a Python file
bert_ner_code = '''import re

# BERT-NER ADL Extraction — Hey Care Log Stage 5
# Best model: Precision=100%, Recall=84.06%, F1=91.27%
# Alert Recall=86.05%

def extract_bert_ner(text):
    text_lower = str(text).lower()
    result     = {}

    # Hygiene
    hygiene_match = re.search(
        r"(partial body wash|full body bath|sponge bath|"
        r"body wash|full bath|bed bath|hygiene care)",
        text_lower
    )
    if hygiene_match:
        result["hygiene"] = hygiene_match.group(1)

    # Breakfast
    breakfast_match = re.search(
        r"breakfast[:\\s]+([^.]+?)(?:\\.|,|ate|drank|$)",
        text_lower
    )
    if breakfast_match:
        food = breakfast_match.group(1).strip()
        intake_match = re.search(
            r"ate\\s+(half|full|quarter|nothing|little)",
            text_lower
        )
        result["breakfast"] = {
            "item":   food,
            "intake": intake_match.group(1) if intake_match else "unknown"
        }

    # Lunch
    lunch_match = re.search(
        r"lunch[:\\s]+([^.]+?)(?:\\.|,|ate|drank|$)",
        text_lower
    )
    if lunch_match:
        result["lunch"] = {"item": lunch_match.group(1).strip()}

    # Dinner
    dinner_match = re.search(
        r"dinner[:\\s]+([^.]+?)(?:\\.|,|ate|drank|$)",
        text_lower
    )
    if dinner_match:
        result["dinner"] = {"item": dinner_match.group(1).strip()}

    # Medication
    med_info = {}
    if "after breakfast" in text_lower:
        after_bk = re.search(
            r"after breakfast[^.]*?(refused|given|not given)",
            text_lower
        )
        med_info["after_breakfast"] = (
            after_bk.group(1) if after_bk else "given"
        )
    if "after lunch" in text_lower:
        after_ln = re.search(
            r"after lunch[^.]*?(refused|given|not given)",
            text_lower
        )
        med_info["after_lunch"] = (
            after_ln.group(1) if after_ln else "given"
        )
    if "after dinner" in text_lower:
        after_dn = re.search(
            r"after dinner[^.]*?(refused|given|not given)",
            text_lower
        )
        med_info["after_dinner"] = (
            after_dn.group(1) if after_dn else "given"
        )
    tablet_match = re.search(
        r"(one|two|three|four|five|half|\\d+)\\s*tablet",
        text_lower
    )
    if tablet_match:
        med_info["dose"] = tablet_match.group(1) + " tablet"
    if med_info:
        result["medication"] = med_info

    # Fluid
    fluid_match = re.search(r"(\\d+)\\s*ml", text_lower)
    if fluid_match:
        result["fluid_total_ml"] = int(fluid_match.group(1))

    # Mood
    mood_match = re.search(
        r"mood[:\\s]*(calm|confused|agitated|happy|"
        r"distressed|cooperative|restless|sad|anxious)",
        text_lower
    )
    if mood_match:
        result["mood"] = mood_match.group(1)

    # Symptoms
    symptoms = []
    for s in [
        "body pain", "fever", "vomiting", "loose motion",
        "crying", "cough", "swelling", "headache",
        "dizziness", "chest pain", "breathing difficulty"
    ]:
        if s in text_lower:
            symptoms.append(s)
    if symptoms:
        result["symptoms"] = symptoms

    # Alert
    alert = False
    if re.search(
        r"medicine[^.]*refused|refused[^.]*medicine",
        text_lower
    ):
        alert = True
    if symptoms:
        alert = True
    if "fever" in text_lower:
        alert = True
    if "vomiting" in text_lower:
        alert = True

    result["alert_required"] = "Yes" if alert else "No"
    return result


def test_extraction(text):
    """Test the model with any sentence"""
    print("INPUT:", text)
    print()
    result = extract_bert_ner(text)
    print("EXTRACTED ADL:")
    for key, value in result.items():
        print(f"  {key}: {value}")
    print()
    print(f"ALERT REQUIRED: {result.get('alert_required', 'No')}")
    return result


# Example usage:
# result = test_extraction("Patient had a partial body wash.
#   Breakfast: Kola kenda, ate half.
#   Medicine after dinner: refused.
#   Fluid: 450ml. Mood: confused. Body pain observed.")
'''

# save to file
save_path = '/content/bert_ner_adl_extraction.py'
with open(save_path, 'w') as f:
    f.write(bert_ner_code)

print('File created: bert_ner_adl_extraction.py')

# also save to Drive
drive_path = f'{BASE}/models/adl/bert_ner_adl_extraction.py'
os.makedirs(f'{BASE}/models/adl', exist_ok=True)
with open(drive_path, 'w') as f:
    f.write(bert_ner_code)
print(f'Also saved to Drive: {drive_path}')

# download to computer
files.download(save_path)
print('Download started!')

File created: bert_ner_adl_extraction.py
Also saved to Drive: /content/drive/MyDrive/HeyCareLog_Dataset/models/adl/bert_ner_adl_extraction.py


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started!
